# [DistilBERT, a distilled verison of BERT: smaller, faster, cheaper and lighter](https://arxiv.org/pdf/1910.01108)

## Introduction

**Gap:** Pre-trained language models like BERT achieve strong performance but are large, slow, and memory-intensive, making them difficult to deploy in real-world or resource-constrained settings.

Prior knowledge distillation approaches mainly focus on task-specific compression **after fine-tuning**, which limits reusability and does not produce a general-purpose efficient pretrained model.

**Improvement**: DistilBERT proposes a method to pretrain a smaller transformer model using knowledge distillation from BERT, producing a foundational model that retains most of BERT’s performance while being significantly more efficient for both inference and downstream fine-tuning.

## Approach

DistilBERT trains a smaller “student” transformer to **mimic** a frozen pretrained BERT “teacher” using **multiple complementary losses**:

* **Language Modeling Loss (MLM):**
The student is trained on masked language modeling, similar to BERT, ensuring it still learns meaningful token-level representations from raw text
* **Distillation Loss (soft-target matching):**
The student matches the teacher’s output distribution (softmax probabilities over vocabulary)
* **Cosine Embedding / Hidden-State Alignment Loss**:
The student is encouraged to align its internal hidden representations with those of the teacher by maximizing cosine similarity between corresponding token-level hidden states.
This acts as an intermediate-level constraint, pushing the student not just to match outputs but also to learn similar feature spaces.


## Result

DistilBERT retains approximately 97% of BERT’s NLU performance while being ~40% smaller and ~60% faster at inference time.

## Application

In [1]:
import pandas as pd
df = pd.read_csv("data/toxicity_en.csv")
df

,text,is_toxic
0,"Elon Musk is a piece of shit, greedy capitalis...",Toxic
1,The senile credit card shrill from Delaware ne...,Toxic
2,He does that a lot -- makes everyone look good...,Toxic
3,F*ck Lizzo,Toxic
4,Epstein and trump were best buds!!! Pedophiles...,Toxic
...,...,...
995,My maternal abuelita taught me how to make pla...,Not Toxic
996,Funnily enough I was looking online last week ...,Not Toxic
997,I can't bear how nice this is.\n \n I guess it...,Not Toxic
998,Going to buy a share of Tesla just to ensure i...,Not Toxic


In [2]:
df.is_toxic.value_counts()

is_toxic
Toxic        501
Not Toxic    499
Name: count, dtype: int64

In [3]:
df['is_toxic'] = (df['is_toxic'] == 'Toxic').astype(int)
df

,text,is_toxic
0,"Elon Musk is a piece of shit, greedy capitalis...",1
1,The senile credit card shrill from Delaware ne...,1
2,He does that a lot -- makes everyone look good...,1
3,F*ck Lizzo,1
4,Epstein and trump were best buds!!! Pedophiles...,1
...,...,...
995,My maternal abuelita taught me how to make pla...,0
996,Funnily enough I was looking online last week ...,0
997,I can't bear how nice this is.\n \n I guess it...,0
998,Going to buy a share of Tesla just to ensure i...,0


In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [5]:
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model (recommended generic classes)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased").to(device)

text = "Replace me by any text you'd like."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Forward pass (no grad for inference)
with torch.no_grad():
    outputs = model(**inputs)

# Outputs
last_hidden_state = outputs.last_hidden_state
pooler_output = outputs.pooler_output  # if available

print(last_hidden_state.shape)
print(pooler_output.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 12, 768])
torch.Size([1, 768])


In [6]:
import torch.nn as nn 

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size=768, output_size=1):
        super().__init__()
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        return self.linear(x)

classifier = ClassifierHead().to(device)
classifier(last_hidden_state.mean(axis=1))

tensor([[-0.2424]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [7]:
from torch.utils.data import Dataset, DataLoader

class ToxicDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df["text"].values
        self.labels = df["is_toxic"].values
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": label
        }

        return item

In [12]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["is_toxic"]  # important for class balance
)

train_dataset = ToxicDataset(train_df, tokenizer, max_length=128)
test_dataset = ToxicDataset(test_df, tokenizer, max_length=128)

from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

batch = next(iter(train_dataloader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)
print()

batch = next(iter(test_dataloader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16])

torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16])


In [15]:
from tqdm import tqdm


def test_bert(model, classifier, criterion):
    model.eval()
    classifier.eval()

    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(test_dataloader, desc="Testing", leave=True)

    with torch.no_grad():
        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].float().to(device)

            # forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            cls_embedding = outputs.last_hidden_state[:, 0, :]
            logits = classifier(cls_embedding).squeeze(-1)

            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(test_dataloader)
    accuracy = correct / total
    return avg_loss, accuracy
    
def train_bert(model, classifier, optimizer, criterion, epochs=30):
    model.train()
    classifier.train()

    for epoch in range(epochs):
        total_loss = 0

        loop = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].float().to(device)

            # forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # CLS token representation
            cls_embedding = outputs.last_hidden_state[:, 0, :]

            logits = classifier(cls_embedding).squeeze(-1)

            loss = criterion(logits, labels)

            # backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_value = loss.item()
            total_loss += loss_value

            loop.set_postfix(loss=loss_value)

        avg_loss = total_loss / len(train_dataloader)
        test_loss, test_acc = test_bert(model, classifier, criterion)
        print(f"Epoch {epoch+1}/{epochs} Train loss: {avg_loss:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}\n")

    return model, classifier

In [21]:
from torch import nn
from torch import optim

LR = 3e-5
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(list(model.parameters()) + list(classifier.parameters()), lr=LR)

In [22]:
train_bert(model, classifier, optimizer, criterion)

Testing: 100%|█████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 28.13it/s, loss=0.114]


Epoch 1/30 Train loss: 0.2444 | Test loss: 0.2348 | Test acc: 0.8933



Testing: 100%|████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 27.96it/s, loss=0.0184]


Epoch 2/30 Train loss: 0.0294 | Test loss: 0.3826 | Test acc: 0.9133



Testing: 100%|█████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 28.80it/s, loss=0.234]


Epoch 3/30 Train loss: 0.0311 | Test loss: 0.3783 | Test acc: 0.8867



Testing: 100%|████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 28.55it/s, loss=0.0448]


Epoch 4/30 Train loss: 0.0093 | Test loss: 0.2820 | Test acc: 0.9333



Epoch 5/30:  44%|████████████████████████▉                               | 24/54 [00:02<00:03,  8.63it/s, loss=9.14e-5]


KeyboardInterrupt: 